In [9]:
library(tidymodels)
library(mlr3)
library(caret)
library(themis)
library(smotefamily)
library(randomForest)
library(ggplot2)
library(mlflow)
library(skimr)
library(data.table)
library(reticulate)
library(data.table)

py_install("scikit-learn")
py_install("imblearn")

library(DoubleML)
library(Matrix)
library(grf)
library(CausalModels)
library(GGally)
library(dplyr)

library(tidyverse)
library(mclust)      # For Gaussian Mixture Models
library(ade4)# For MCA on categorical data  
library(mvtnorm)
library(stringr)
library(fastDummies)
options(warn=-1)


Using virtual environment '/home/voltsy/.virtualenvs/r-reticulate' ...


+ /home/voltsy/.virtualenvs/r-reticulate/bin/python -m pip install --upgrade --no-user scikit-learn



Using virtual environment '/home/voltsy/.virtualenvs/r-reticulate' ...


+ /home/voltsy/.virtualenvs/r-reticulate/bin/python -m pip install --upgrade --no-user imblearn



In [1]:
train_processed <- read.csv("data/train_with_regimes.csv")
test_processed <- read.csv("data/test_with_regimes.csv")

In [7]:
train_processed

PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,⋯,OneHotKMeans_Regime2_Prob,OneHotKMeans_Regime3_Prob,CryoRegime_Prob,LuxuryRegime_Prob,FamilyRegime_Prob,SoloRegime_Prob,Ensemble_Regime1_Prob,Ensemble_Regime2_Prob,Ensemble_Regime3_Prob,Predicted_Regime
<chr>,<chr>,<lgl>,<chr>,<chr>,<int>,<lgl>,<int>,<int>,<int>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
0001_01,Europa,FALSE,B/0/P,TRAPPIST-1e,39,FALSE,0,0,0,⋯,0.3679421,0.3094876,0.40000000,0.0000000,0.06666667,0.53333333,0.2583857,0.6072647,0.1343496,2
0002_01,Earth,FALSE,F/0/S,TRAPPIST-1e,24,FALSE,109,9,25,⋯,0.2775218,0.3839503,0.06451613,0.3548387,0.06451613,0.51612903,0.5022565,0.2248386,0.2729049,1
0003_01,Europa,FALSE,A/0/S,TRAPPIST-1e,58,TRUE,43,3576,0,⋯,0.3140453,0.3754807,0.05555556,0.5555556,0.22222222,0.16666667,0.1156152,0.2746109,0.6097740,3
0003_02,Europa,FALSE,A/0/S,TRAPPIST-1e,33,FALSE,0,1283,371,⋯,0.3037932,0.3955071,0.07407407,0.4074074,0.29629630,0.22222222,0.1205132,0.2286003,0.6508865,3
0004_01,Earth,FALSE,F/1/S,TRAPPIST-1e,16,FALSE,303,70,151,⋯,0.2870296,0.3740178,0.06451613,0.3548387,0.06451613,0.51612903,0.6010666,0.2278685,0.1710648,1
0005_01,Earth,FALSE,F/0/P,PSO J318.5-22,44,FALSE,0,483,0,⋯,0.3378151,0.3063373,0.08333333,0.1666667,0.08333333,0.66666667,0.1656396,0.2000404,0.6343201,3
0006_01,Earth,FALSE,F/2/S,TRAPPIST-1e,26,FALSE,42,1539,3,⋯,0.2775218,0.3839503,0.09090909,0.2727273,0.36363636,0.27272727,0.3616744,0.2443530,0.3939726,3
0006_02,Earth,TRUE,G/0/S,TRAPPIST-1e,28,FALSE,0,0,0,⋯,0.3169429,0.3974845,0.56250000,0.0000000,0.25000000,0.18750000,0.2698751,0.5242941,0.2058308,2
0007_01,Earth,FALSE,F/3/S,TRAPPIST-1e,35,FALSE,0,785,17,⋯,0.2775218,0.3839503,0.08333333,0.1666667,0.08333333,0.66666667,0.1627343,0.1697946,0.6674711,3


In [3]:
set.seed(42)

target_col <- "Transported"
idx <- sample(nrow(train_processed), size=0.8 * nrow(train_processed))

train_data <- train_processed[idx, ]
test_data  <- train_processed[-idx, ]

# y vectors
y_train <- train_data[[target_col]]
y_test  <- test_data[[target_col]]

# remove target from X
train_data[[target_col]] <- NULL
test_data[[target_col]] <- NULL

# convert to sparse matrices
X_train <- Matrix::sparse.model.matrix(
  ~ . - 1,
  data = train_data
)

X_test <- Matrix::sparse.model.matrix(
  ~ . - 1,
  data = test_data
)

In [10]:
# ============================================
# Imputation + Encoding + SMOTE
# ============================================

library(data.table)
library(reticulate)

# sklearn
sklearn <- import("sklearn")
imblearn <- import("imblearn")

SimpleImputer <- sklearn$impute$SimpleImputer
LabelEncoder <- sklearn$preprocessing$LabelEncoder
SMOTE <- imblearn$over_sampling$SMOTE

# ============================================
# Copy datasets
# ============================================

train_df <- copy(train_processed)
test_df <- copy(test_processed)

# ============================================
# Target column
# ============================================

target_col <- "Transported"

# ============================================
# Remove identifiers
# ============================================

drop_cols <- intersect(
  c("PassengerId", "Name"),
  names(train_df)
)

if (length(drop_cols) > 0) {

  train_df[, (drop_cols) := NULL]

  test_drop <- intersect(
    drop_cols,
    names(test_df)
  )

  if (length(test_drop) > 0) {
    test_df[, (test_drop) := NULL]
  }
}

# ============================================
# Detect column types
# ============================================

categorical_cols <- names(train_df)[
  sapply(train_df, function(x)
    is.character(x) ||
    is.factor(x) ||
    is.logical(x))
]

categorical_cols <- setdiff(
  categorical_cols,
  target_col
)

numeric_cols <- names(train_df)[
  sapply(train_df, is.numeric)
]

numeric_cols <- setdiff(
  numeric_cols,
  target_col
)

# ============================================
# NUMERIC IMPUTATION
# ============================================

num_imputer <- SimpleImputer(
  strategy = "median"
)

train_num <- as.matrix(
  train_df[, ..numeric_cols]
)

test_num <- as.matrix(
  test_df[, ..numeric_cols]
)

mode(train_num) <- "numeric"
mode(test_num) <- "numeric"

train_num[
  is.infinite(train_num)
] <- NA

test_num[
  is.infinite(test_num)
] <- NA

num_imputer$fit(train_num)

train_num_imputed <- num_imputer$transform(
  train_num
)

test_num_imputed <- num_imputer$transform(
  test_num
)

train_num_imputed <- as.data.table(
  train_num_imputed
)

test_num_imputed <- as.data.table(
  test_num_imputed
)

setnames(
  train_num_imputed,
  numeric_cols
)

setnames(
  test_num_imputed,
  numeric_cols
)

train_df[, (numeric_cols) := train_num_imputed]
test_df[, (numeric_cols) := test_num_imputed]

# ============================================
# CATEGORICAL CLEANING
# ============================================

for (col in categorical_cols) {

  # TRAIN
  train_vals <- as.character(
    train_df[[col]]
  )

  train_vals[is.na(train_vals)] <- "Missing"

  train_vals <- iconv(
    train_vals,
    from = "",
    to = "UTF-8",
    sub = ""
  )

  train_vals <- gsub(
    "[[:cntrl:]]",
    "",
    train_vals
  )

  train_vals <- substr(
    train_vals,
    1,
    100
  )

  train_vals[
    trimws(train_vals) == ""
  ] <- "Missing"

  train_df[[col]] <- train_vals

  # TEST
  test_vals <- as.character(
    test_df[[col]]
  )

  test_vals[is.na(test_vals)] <- "Missing"

  test_vals <- iconv(
    test_vals,
    from = "",
    to = "UTF-8",
    sub = ""
  )

  test_vals <- gsub(
    "[[:cntrl:]]",
    "",
    test_vals
  )

  test_vals <- substr(
    test_vals,
    1,
    100
  )

  test_vals[
    trimws(test_vals) == ""
  ] <- "Missing"

  test_df[[col]] <- test_vals
}

# ============================================
# LABEL ENCODING
# ============================================

label_encoders <- list()

for (col in categorical_cols) {

  le <- LabelEncoder()

  combined_vals <- c(
    as.character(train_df[[col]]),
    as.character(test_df[[col]])
  )

  le$fit(combined_vals)

  train_df[[col]] <- as.integer(
    le$transform(
      as.character(train_df[[col]])
    )
  )

  test_df[[col]] <- as.integer(
    le$transform(
      as.character(test_df[[col]])
    )
  )

  label_encoders[[col]] <- le
}

# ============================================
# TRAIN MATRICES
# ============================================

X_train <- train_df[
  ,
  !target_col,
  with = FALSE
]

y_train <- train_df[[target_col]]

if (is.logical(y_train)) {
  y_train <- as.integer(y_train)
}

# ============================================
# SMOTE
# ============================================

smote <- SMOTE(
  sampling_strategy = "auto",
  random_state = 42L,
  k_neighbors = 5L
)

smote_result <- smote$fit_resample(
  as.matrix(X_train),
  y_train
)

X_train_balanced <- as.data.table(
  py_to_r(smote_result[[1]])
)

y_train_balanced <- py_to_r(
  smote_result[[2]]
)

X_train_balanced[
  ,
  Transported := y_train_balanced
]

# ============================================
# Final outputs
# ============================================

train_balanced <- X_train_balanced
test_final <- test_df

cat(
  "\nOriginal Train Rows:",
  nrow(train_df),
  "\nBalanced Train Rows:",
  nrow(train_balanced),
  "\nTest Rows:",
  nrow(test_final),
  "\n"
)

ERROR: Error in `:=`((drop_cols), NULL): Check that is.data.table(DT) == TRUE. Otherwise, :=, `:=`(...) and let(...) are defined for use in j, once only and in particular ways. Note that namespace-qualification like data.table::`:=`(...) is not supported. See help(":=").


In [ ]:
mlflow_set_tracking_uri("mlruns")

mlflow_experiment("spaceship_titanic_regime_models")


with(mlflow_start_run(run_name = "Logistic_Regression"), {

  lr_model <- cv.glmnet(
    x = X_train_smote,
    y = y_train_smote_num,
    family = "binomial",
    alpha = 0
  )

  preds_prob <- predict(
    lr_model,
    X_valid,
    s = "lambda.min",
    type = "response"
  )

  preds <- ifelse(preds_prob > 0.5, 1, 0)

  acc <- Accuracy(y_pred = preds, y_true = y_valid_num)

  mlflow_log_metric("accuracy", acc)

  mlflow_log_param("model", "glmnet")

  saveRDS(lr_model, "lr_model.rds")

  mlflow_log_artifact("lr_model.rds")
})


with(mlflow_start_run(run_name = "Random_Forest"), {

  rf_model <- randomForest(
    x = as.matrix(X_train_smote),
    y = y_train_smote,
    ntree = 200,
    maxnodes = 30
  )

  preds <- predict(
    rf_model,
    as.matrix(X_valid)
  )

  acc <- Accuracy(
    y_pred = preds,
    y_true = y_valid
  )

  mlflow_log_metric("accuracy", acc)

  mlflow_log_param("ntree", 200)

  saveRDS(rf_model, "rf_model.rds")

  mlflow_log_artifact("rf_model.rds")
})

with(mlflow_start_run(run_name = "XGBoost"), {

  dtrain <- xgb.DMatrix(
    data = X_train_smote,
    label = y_train_smote_num
  )

  dvalid <- xgb.DMatrix(
    data = X_valid,
    label = y_valid_num
  )

  xgb_model <- xgboost(
    data = dtrain,
    nrounds = 300,
    max_depth = 6,
    eta = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    objective = "binary:logistic",
    eval_metric = "logloss",
    tree_method = "hist",
    nthread = 2,
    verbose = 0
  )

  preds_prob <- predict(
    xgb_model,
    dvalid
  )

  preds <- ifelse(preds_prob > 0.5, 1, 0)

  acc <- Accuracy(
    y_pred = preds,
    y_true = y_valid_num
  )

  mlflow_log_metric("accuracy", acc)

  mlflow_log_param("max_depth", 6)
  mlflow_log_param("eta", 0.05)

  xgb.save(
    xgb_model,
    "xgb_model.json"
  )

  mlflow_log_artifact("xgb_model.json")
})



with(mlflow_start_run(run_name = "LightGBM"), {

  lgb_train <- lgb.Dataset(
    data = X_train_smote,
    label = y_train_smote_num
  )

  lgb_model <- lightgbm(
    data = lgb_train,
    objective = "binary",
    learning_rate = 0.05,
    num_leaves = 31,
    nrounds = 300,
    num_threads = 2,
    verbose = -1
  )

  preds_prob <- predict(
    lgb_model,
    X_valid
  )

  preds <- ifelse(preds_prob > 0.5, 1, 0)

  acc <- Accuracy(
    y_pred = preds,
    y_true = y_valid_num
  )

  mlflow_log_metric("accuracy", acc)

  mlflow_log_param("num_leaves", 31)

  lgb.save(
    lgb_model,
    "lgb_model.txt"
  )

  mlflow_log_artifact("lgb_model.txt")
})



with(mlflow_start_run(run_name = "CatBoost"), {

  train_pool <- catboost.load_pool(
    data = as.data.frame(as.matrix(X_train_smote)),
    label = y_train_smote_num
  )

  valid_pool <- catboost.load_pool(
    data = as.data.frame(as.matrix(X_valid)),
    label = y_valid_num
  )

  cat_model <- catboost.train(
    learn_pool = train_pool,
    params = list(
      loss_function = "Logloss",
      iterations = 300,
      depth = 6,
      learning_rate = 0.05,
      thread_count = 2
    )
  )

  preds_prob <- catboost.predict(
    cat_model,
    valid_pool,
    prediction_type = "Probability"
  )

  preds <- ifelse(preds_prob > 0.5, 1, 0)

  acc <- Accuracy(
    y_pred = preds,
    y_true = y_valid_num
  )

  mlflow_log_metric("accuracy", acc)

  mlflow_log_param("depth", 6)

  catboost.save_model(
    cat_model,
    "cat_model.cbm"
  )

  mlflow_log_artifact("cat_model.cbm")
})

test_ids <- test_df$PassengerId

if("Transported" %in% names(test_df)) {
  test_df$Transported <- NULL
}

for(col in names(test_df)) {

  if(is.character(test_df[[col]])) {
    test_df[[col]] <- as.factor(test_df[[col]])
  }
}

X_test <- sparse.model.matrix(
  ~ . - 1,
  data = test_df
)


dtest <- xgb.DMatrix(X_test)

xgb_preds_prob <- predict(
  xgb_model,
  dtest
)

xgb_preds <- xgb_preds_prob > 0.5

submission <- data.frame(
  PassengerId = test_ids,
  Transported = xgb_preds
)

write.csv(
  submission,
  "submission.csv",
  row.names = FALSE
)

print(head(submission))


cat("\nTraining Complete.\n")
cat("Submission file saved as submission.csv\n")